[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_GJRGARCH/VaR_CEE_GJRGARCH.ipynb)

# VaR_CEE_GJRGARCH

Implements GJR-GARCH(1,1) with skewed-t distribution for VaR and ES forecasting
using a rolling 250-day window with daily refitting. Falls back to normal distribution
if skewed-t fails to converge.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
!pip install -q arch

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings("ignore")
%matplotlib inline

# ── Configuration ──────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
OOS_START = "2018-01-01"
VAR_ALPHAS = [0.01, 0.025, 0.05]
ES_ALPHA = 0.025
ROLLING_WINDOW = 250
STRIDE = 1  # R1: full daily run
SEED = 42

def get_all_series():
    series = []
    for country, info in MARKETS.items():
        series.append((country, info["index"], f"{info['index']}_ret", "index"))
        series.append((country, info["fx"], f"{info['fx']}_ret", "fx"))
    return series

W = ROLLING_WINDOW
np.random.seed(SEED)

OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

## Upload Data

Upload the 10 raw CSV files for index and FX series.

In [ ]:
from google.colab import files
print("Upload the 10 raw CSV files (BET.csv, WIG20.csv, PX.csv, BUX.csv, SOFIX.csv,")
print("EURRON.csv, EURPLN.csv, EURCZK.csv, EURHUF.csv, USDBGN.csv):")
uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} files")

# Compute log returns
returns_dict = {}
for country, raw_id, series_id, stype in get_all_series():
    csv_file = f"{raw_id}.csv"
    if csv_file not in uploaded:
        print(f"  WARNING: {csv_file} not found, skipping {series_id}")
        continue
    df = pd.read_csv(csv_file, parse_dates=["Date"], index_col="Date").sort_index()
    log_ret = np.log(df["Close"] / df["Close"].shift(1)).dropna()
    log_ret.name = series_id
    returns_dict[series_id] = log_ret
    print(f"  {series_id}: {len(log_ret)} obs")
print(f"\nTotal: {len(returns_dict)} series")

In [ ]:
def run_gjr_garch(returns, oos_dates):
    """GJR-GARCH(1,1) with skewed-t, rolling 250-day window, daily refit."""
    from arch import arch_model
    results = []
    for i, date in enumerate(oos_dates):
        loc = returns.index.get_loc(date)
        if loc < W:
            continue
        train = returns.iloc[loc - W:loc] * 100

        try:
            am = arch_model(train, mean="Constant", vol="GARCH",
                            p=1, o=1, q=1, dist="skewt")
            res = am.fit(disp="off", show_warning=False)
        except Exception:
            try:
                am = arch_model(train, mean="Constant", vol="GARCH",
                                p=1, o=1, q=1, dist="normal")
                res = am.fit(disp="off", show_warning=False)
            except Exception:
                continue

        try:
            fcst = res.forecast(horizon=1, reindex=False)
            mu = fcst.mean.iloc[-1, 0] / 100
            sigma = np.sqrt(fcst.variance.iloc[-1, 0]) / 100
            std_resids = res.std_resid.dropna()
            y_true = returns.iloc[loc]
            row = {"date": date, "y_true": y_true}
            for alpha in VAR_ALPHAS:
                q_alpha = np.quantile(std_resids, alpha)
                row[f"var_{alpha}"] = mu + sigma * q_alpha
            q_es = np.quantile(std_resids, ES_ALPHA)
            tail_r = std_resids[std_resids <= q_es]
            row["es_2p5pct"] = mu + sigma * (np.mean(tail_r) if len(tail_r) > 0 else q_es)
            results.append(row)
        except Exception:
            continue

        if (i + 1) % 100 == 0:
            print(f"    GARCH [{i+1}/{len(oos_dates)}]")

    return pd.DataFrame(results)

In [ ]:
print("=" * 60)
print("GJR-GARCH(1,1) (rolling 250-day window, daily refit)")
print("=" * 60)

all_results = {}
for country, raw_id, series_id, stype in get_all_series():
    if series_id not in returns_dict:
        continue
    returns = returns_dict[series_id]
    oos_mask = returns.index >= pd.Timestamp(OOS_START)
    oos_dates = returns.index[oos_mask][::STRIDE]
    print(f"\n[{series_id}] {len(returns)} obs, {len(oos_dates)} OOS dates")

    t0 = time.time()
    df_result = run_gjr_garch(returns, oos_dates)
    col_map = {f"var_{a}": f"var_{str(a).replace('0.', '')}pct" for a in VAR_ALPHAS}
    df_result = df_result.rename(columns=col_map)
    out_path = OUT_DIR / f"GJR-GARCH_{series_id}.csv"
    df_result.to_csv(out_path, index=False, float_format="%.8f")
    all_results[series_id] = df_result
    print(f"  Saved {out_path.name} ({len(df_result)} rows) in {time.time()-t0:.1f}s")

print("\n=== GJR-GARCH complete ===")

In [ ]:
for country, raw_id, series_id, stype in get_all_series():
    if stype != "index" or series_id not in all_results:
        continue
    df_plot = all_results[series_id]
    var_col = "var_01pct"
    if var_col not in df_plot.columns:
        continue
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df_plot["date"], df_plot["y_true"], linewidth=0.4, color="black", label="Returns")
    ax.plot(df_plot["date"], df_plot[var_col], linewidth=0.6, color="blue", linestyle="--", label="GARCH VaR 1%")
    breaches = df_plot["y_true"] < df_plot[var_col]
    if breaches.any():
        ax.scatter(df_plot.loc[breaches, "date"], df_plot.loc[breaches, "y_true"],
                   color="red", s=10, zorder=5, alpha=0.7, label="Breach")
    n_b = breaches.sum()
    ax.set_title(f"GJR-GARCH(1,1) VaR 1% \u2014 {series_id} (breaches: {n_b}/{len(df_plot)} = {100*n_b/len(df_plot):.1f}%)")
    ax.set_ylabel("Log returns")
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
    ax.axhline(y=0, color="gray", linewidth=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
from google.colab import files
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT_DIR.glob("GJR-GARCH_*.csv")):
        zf.write(f, f.name)
zip_buffer.seek(0)
with open("GJRGARCH_results.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("GJRGARCH_results.zip")